# Embedding + 단순 검색, 직접 만들어보기
- 벡터 DB 를 쓰기 전에 **numpy 만으로** 작은 RAG 검색기를 만들어봅니다. 원리가 손에 잡혀야 다음 노트북의 진짜 벡터 DB가 이해됩니다.

### 참고자료
- https://docs.langchain.com/oss/python/integrations/embeddings
- https://wikidocs.net/231431

### Embedding이란?

텍스트를 고차원 벡터로 변환합니다.

```
"오늘 김밥을 먹었다."  →  [0.12, -0.34, 0.56, ...] (1024차원)
```

벡터로 변환하면 **의미적 유사성**을 계산할 수 있습니다.

### 임베딩 모델 선택

| 모델 | 차원 | 특징 |
|------|------|------|
| **Ollama bge-m3** | 1024 | 무료, 한국어 강력, 로컬 실행 (`ollama pull bge-m3`) |
| Google gemini-embedding-2 | 3072 | API 키 필요, 무료 할당량 제공 |
| OpenAI text-embedding-3-small | 1536 | 가성비 우수, 영어 특화 |

## 1. 환경 준비

## (1) 라이브러리 설치

처음 실행하는 환경이라면 아래 셀의 주석을 해제하고 실행합니다. 이미 `pyproject.toml` 또는 `requirements.txt`로 설치했다면 실행하지 않아도 됩니다.


In [ ]:
# 필요한 라이브러리 설치
# uv add -qU langchain langchain-community langchain-text-splitters langchain-openai langchain-experimental pypdfium2 pypdf scikit-learn

## (2) API Key 설정

API 키는 코드에 직접 작성하지 않습니다. 권장 방식은 `.env` 파일에 저장하는 것입니다.

```text
# OpenAI를 사용할 때
OPENAI_API_KEY=sk-...

# Gemini를 사용할 때
GOOGLE_API_KEY=...

```


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

print("OPENAI_API_KEY:", "있음" if os.getenv("OPENAI_API_KEY") else "없음")
print("GOOGLE_API_KEY:", "있음" if os.getenv("GOOGLE_API_KEY") else "없음")

## 2. 문서 준비 + 분할

In [ ]:
SAMPLE = """## 우주지사 근무 규정

### 1. 화성 지사 출근
화성 지사 근무자는 지구 표준시 기준 오전 9시까지 메타버스 출근 체크를 완료해야 한다. 출근 체크는 사내 근태 시스템의 "화성 지사 원격 출근" 메뉴에서 진행하며, 위치 인증과 생체 인증을 모두 통과해야 정상 출근으로 인정된다.

산소 농도 경고가 발생한 경우에는 출근 체크보다 안전 확인 보고서를 먼저 제출해야 한다. 안전 확인 보고서에는 현재 위치, 산소 농도 수치, 근무 가능 여부, 대피 필요 여부를 기록해야 한다. 산소 농도가 기준치 이하로 10분 이상 유지되면 해당 근무자는 자동으로 비상 대기 상태로 전환된다.

화성 모래폭풍 경보가 발령된 날에는 지사장이 재택근무 전환 여부를 공지한다. 재택근무로 전환된 경우에도 오전 10시까지 업무 계획을 팀 채널에 공유해야 하며, 긴급 연락을 받을 수 있도록 메신저 상태를 온라인으로 유지해야 한다.

### 2. 우주복 장비 대여
외부 기지 이동 시 우주복, 산소팩, 자기부착 신발을 반드시 대여해야 한다. 장비 대여는 이동 예정 시간 최소 1시간 전까지 장비 관리 시스템에서 신청해야 하며, 신청서에는 이동 목적, 예상 이동 경로, 복귀 예정 시간을 입력해야 한다.

우주복은 사용 후 18시까지 장비실에 반납해야 한다. 반납 시에는 외부 손상 여부, 산소 밸브 상태, 통신 모듈 작동 여부를 점검표에 기록해야 한다. 우주복에 균열이 있거나 통신 모듈 오류가 발견되면 즉시 장비 담당자에게 보고해야 하며, 임의로 수리해서는 안 된다.

산소팩 잔량이 20% 미만이면 즉시 교체 신청을 해야 한다. 산소팩 잔량이 10% 이하로 떨어진 상태에서 외부 이동을 계속하는 것은 중대한 안전 규정 위반으로 간주된다. 자기부착 신발은 기지 외부에서는 항상 활성화해야 하며, 실내 복귀 후에는 바닥 손상을 방지하기 위해 비활성화해야 한다.

### 3. 화성 회의실 예약
화성 회의실은 최소 2시간 전까지 예약해야 한다. 예약은 사내 캘린더의 "화성 지사 회의실" 메뉴에서 진행하며, 회의 목적, 참석자 수, 예상 소요 시간, 필요한 장비를 함께 입력해야 한다. 회의실은 기본 1시간 단위로 예약할 수 있으며, 3시간을 초과하는 회의는 지사장 승인이 필요하다.

6명 이상 참석하는 회의는 산소 소비량 계산을 위해 참석자 명단을 함께 등록해야 한다. 참석자가 외부 방문자인 경우에는 방문 목적과 소속 기관을 추가로 입력해야 하며, 보안 구역 회의실은 외부 방문자 예약이 제한된다.

회의 시작 10분 전까지 입실하지 않으면 예약은 자동 취소될 수 있다. 회의 종료 후에는 공용 화면, 홀로그램 프로젝터, 산소 조절 장치를 초기 상태로 되돌려야 한다. 회의 중 산소 농도 알림이 발생하면 회의를 즉시 중단하고, 참석자는 가장 가까운 안전 구역으로 이동해야 한다.
"""

print(f"문서 길이: {len(SAMPLE)} 자")

In [ ]:


print(f"청크 {len(chunks)} 개")
for i, c in enumerate(chunks):
    print(f"  [{i}] {c[:60]}...")

## 3. 청크 임베딩

In [ ]:


chunk_vectors = np.array(embeddings.embed_documents(chunks))
print(f"임베딩 행렬 shape: {chunk_vectors.shape}")

## 4. 검색 함수, 코사인 top-k

In [ ]:
def normalize(v):
    """벡터 단위화."""
    return v / np.linalg.norm(v, axis=-1, keepdims=True)


# 미리 정규화해두면 dot product 가 곧 코사인 유사도
chunk_vectors_n = normalize(chunk_vectors)


def search(query: str, k: int = 3):
    q_vec = np.array(embeddings.embed_query(query))
    q_vec_n = q_vec / np.linalg.norm(q_vec)
    # 모든 청크와의 유사도 한 번에 계산
    sims = chunk_vectors_n @ q_vec_n
    # 상위 k 개 인덱스
    top_idx = np.argsort(sims)[::-1][:k]
    return [(chunks[i], float(sims[i])) for i in top_idx]


for q in [" ", " ", " "]:
    print(f"\n질문: {q}")
    for chunk, score in search(q, k= ):
        print(f"  {score:.3f}  {chunk[:80]}")

## 5. 메타데이터 함께 저장
- 실무에서는 청크 + "출처(파일명·페이지)" 메타데이터를 같이 저장
- 보통 원문 로더 단계에서 metadata 를 붙여 두고, 분할 시점에 청크가 상속받게 함

In [ ]:
docs_with_meta = []
for i, c in enumerate(chunks):
    # 메타 데이터


    docs_with_meta.append({
        "id": i,
        "text": c,
        "metadata": {
            "section": section,
            "source": "space_branch_policy.md",
        },
    })


def search_with_meta(query: str, k: int = 3, filter_section: str | None = None):
    q_vec = embeddings.embed_query(query)
    q_vec_n = np.array(q_vec) / np.linalg.norm(q_vec)
    sims = chunk_vectors_n @ q_vec_n

    candidates = []
    for i, sim in enumerate(sims):
        d = docs_with_meta[i]
        if filter_section and d["metadata"]["section"] != filter_section:
            continue
        candidates.append((d, float(sim)))

    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[:k]


# 장비대여 섹션 안에서만 검색
results = search_with_meta(

)

for d, score in results:
    print(f"{score:.3f}  [{d['metadata']['section']}]  {d['text'][:80]}")

## 6. 정리

- 임베딩은 한 번에 배치로 (`embed_documents`)
- 검색은 코사인 유사도 (정규화 후 dot product)
- 메타데이터로 출처·섹션 필터 가능
- 다음 노트북에서 이걸 LLM 답변 생성과 연결

### [실습]
1. 다른 청크 / 다른 질문 추가 후 검색, 의도한 청크가 위로 오는지.
2. `k` 값을 1 / 5 / 10 으로 바꿔 결과 비교.
3. 한국어 문서 50개를 임베딩해 검색 정확도 측정 (정답 청크가 top-3 안에 들어오는지).
4. 메타데이터 필터를 "회의실예약" 같은 조건으로 확장.